# Learn2Branch Set Cover 100k Colab Run

This notebook follows the official Ecole Learn2Branch pipeline as closely as possible:

- official Learn2Branch set cover generator
- official Ecole `02_generate_dataset.py` sample format
- official Ecole `03_train_gnn.py` model, dataloader, padding, loss, scheduler, and checkpointing
- 100,000 train expert samples by default
- GPU training when Colab provides CUDA

Run this on a Colab GPU runtime. The first cell installs Conda and restarts the runtime.

In [ ]:
# Cell 1: Install Conda in Colab
# After this cell finishes, Colab will restart the runtime. Then continue from Cell 2.
!pip -q install condacolab
import condacolab
condacolab.install()

In [ ]:
# Cell 2: Install Dependencies and Clone Official Repos
# Ecole is installed from conda-forge. Torch/PyG are installed with pip so Colab can use CUDA wheels.
!conda install -y -c conda-forge ecole pyscipopt numpy scipy pandas matplotlib
!pip -q install torch torch-geometric

%cd /content
!test -d learn2branch-ecole || git clone https://github.com/ds4dm/learn2branch-ecole.git
!test -d learn2branch || git clone https://github.com/ds4dm/learn2branch.git

import ecole, torch, torch_geometric, pyscipopt
print('ecole:', ecole.__version__)
print('torch:', torch.__version__)
print('torch_geometric:', torch_geometric.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

In [ ]:
# Cell 3: Configuration
from pathlib import Path
import torch

ROOT = Path('/content')
ECOLE_REPO = ROOT / 'learn2branch-ecole'
ORIGINAL_REPO = ROOT / 'learn2branch'

# Official set cover size from Learn2Branch: 500 rows, 1000 cols, 0.05 density.
NROWS = 500
NCOLS = 1000
DENSITY = 0.05

# Full Learn2Branch-style instance counts are large. These defaults keep the official scale.
# Lower them only if Colab storage/time becomes an issue.
N_TRAIN_INSTANCES = 10_000
N_VALID_INSTANCES = 2_000
N_TEST_INSTANCES = 2_000

# Full supervised sample budget requested.
TRAIN_SAMPLES = 100_000
VALID_SAMPLES = 20_000
TEST_SAMPLES = 0

# Official sampler uses 0.05. This means labels are sparse but matches Learn2Branch/Ecole.
NODE_RECORD_PROB = 0.05
SAMPLE_TIME_LIMIT = 3600
N_JOBS = 4

# Official trainer defaults.
SEED = 0
MAX_EPOCHS = 1000
EPOCH_SAMPLES = 10_000
BATCH_SIZE = 32
GPU_ID = 0 if torch.cuda.is_available() else -1

print('GPU_ID:', GPU_ID)

In [ ]:
# Cell 4: Patch Official Ecole Scripts for Modern PyTorch and Configurable Scale
# The model/training logic stays official. This only adds CLI knobs and PyTorch API compatibility.
from pathlib import Path

dataset_py = ECOLE_REPO / '02_generate_dataset.py'
train_py = ECOLE_REPO / '03_train_gnn.py'
utilities_py = ECOLE_REPO / 'utilities.py'

text = dataset_py.read_text()
if "--train-size" not in text:
    text = text.replace(
        "    parser.add_argument(\n        '-j', '--njobs',\n        help='Number of parallel jobs.',\n        type=int,\n        default=1,\n    )",
        "    parser.add_argument(\n        '-j', '--njobs',\n        help='Number of parallel jobs.',\n        type=int,\n        default=1,\n    )\n    parser.add_argument('--train-size', type=int, default=100000)\n    parser.add_argument('--valid-size', type=int, default=20000)\n    parser.add_argument('--test-size', type=int, default=20000)\n    parser.add_argument('--node-record-prob', type=float, default=0.05)\n    parser.add_argument('--time-limit', type=float, default=None)"
    )
    text = text.replace("    train_size = 100000\n    valid_size = 20000\n    test_size = 20000\n    node_record_prob = 0.05", "    train_size = args.train_size\n    valid_size = args.valid_size\n    test_size = args.test_size\n    node_record_prob = args.node_record_prob")
    text = text.replace("    print(f\"{len(instances_train)} train instances for {train_size} samples\")", "    if args.time_limit is not None:\n        time_limit = args.time_limit\n\n    print(f\"{len(instances_train)} train instances for {train_size} samples\")")
    text = text.replace("collect_samples(instances_valid, out_dir + '/valid', rng, test_size,", "collect_samples(instances_valid, out_dir + '/valid', rng, valid_size,")
    dataset_py.write_text(text)

text = train_py.read_text()
if "--max-epochs" not in text:
    text = text.replace(
        "    parser.add_argument(\n        '-g', '--gpu',\n        help='CUDA GPU id (-1 for CPU).',\n        type=int,\n        default=0,\n    )",
        "    parser.add_argument(\n        '-g', '--gpu',\n        help='CUDA GPU id (-1 for CPU).',\n        type=int,\n        default=0,\n    )\n    parser.add_argument('--max-epochs', type=int, default=1000)\n    parser.add_argument('--epoch-samples', type=int, default=10000)\n    parser.add_argument('--batch-size', type=int, default=32)\n    parser.add_argument('--pretrain-batch-size', type=int, default=128)\n    parser.add_argument('--valid-batch-size', type=int, default=128)"
    )
    text = text.replace("    max_epochs = 1000\n    batch_size = 32\n    pretrain_batch_size = 128\n    valid_batch_size = 128", "    max_epochs = args.max_epochs\n    batch_size = args.batch_size\n    pretrain_batch_size = args.pretrain_batch_size\n    valid_batch_size = args.valid_batch_size")
    text = text.replace("int(np.floor(10000/batch_size))*batch_size", "int(np.floor(args.epoch_samples/batch_size))*batch_size")
text = text.replace("Scheduler(optimizer, mode='min', patience=10, factor=0.2, verbose=True)", "Scheduler(optimizer, mode='min', patience=10, factor=0.2)")
train_py.write_text(text)

text = utilities_py.read_text()
if "_is_better" not in text[text.find('class Scheduler'):]:
    text = text.replace("        if self.is_better(current, self.best):", "        is_better = self.is_better if hasattr(self, 'is_better') else self._is_better\n        if is_better(current, self.best):")
    utilities_py.write_text(text)

!python -m py_compile /content/learn2branch-ecole/02_generate_dataset.py /content/learn2branch-ecole/03_train_gnn.py /content/learn2branch-ecole/utilities.py
print('patches applied')

In [ ]:
# Cell 5: Generate Official Learn2Branch Set Cover Instances
# Uses original ds4dm/learn2branch generate_setcover(...), saved into the official Ecole folder layout.
import importlib.util
import sys
import os
import numpy as np

sys.path.insert(0, str(ORIGINAL_REPO))
spec = importlib.util.spec_from_file_location('l2b_generator', ORIGINAL_REPO / '01_generate_instances.py')
l2b_generator = importlib.util.module_from_spec(spec)
spec.loader.exec_module(l2b_generator)

def generate_split(split_name, n_instances, seed):
    out_dir = ECOLE_REPO / 'data' / 'instances' / 'setcover' / f'{split_name}_{NROWS}r_{NCOLS}c_{DENSITY}d'
    out_dir.mkdir(parents=True, exist_ok=True)
    existing = len(list(out_dir.glob('instance_*.lp')))
    if existing >= n_instances:
        print(f'{split_name}: already has {existing}/{n_instances}')
        return
    rng = np.random.RandomState(seed)
    for i in range(existing, n_instances):
        filename = out_dir / f'instance_{i + 1}.lp'
        l2b_generator.generate_setcover(NROWS, NCOLS, DENSITY, str(filename), rng, max_coef=100)
        if (i + 1) % 100 == 0 or (i + 1) == n_instances:
            print(f'{split_name}: generated {i + 1}/{n_instances}', flush=True)

generate_split('train', N_TRAIN_INSTANCES, SEED)
generate_split('valid', N_VALID_INSTANCES, SEED + 1)
generate_split('test', N_TEST_INSTANCES, SEED + 2)

!find /content/learn2branch-ecole/data/instances/setcover -maxdepth 2 -name '*.lp' | sed 's#/content/learn2branch-ecole/data/instances/setcover/##' | cut -d/ -f1 | sort | uniq -c

In [ ]:
# Cell 6: Generate Official Ecole Supervised Dataset Samples
# This is the expensive CPU-bound step. It creates gzip pickle samples in data/samples/setcover/500r_1000c_0.05d/.
%cd /content/learn2branch-ecole
!python 02_generate_dataset.py setcover -j {N_JOBS} --train-size {TRAIN_SAMPLES} --valid-size {VALID_SAMPLES} --test-size {TEST_SAMPLES} --node-record-prob {NODE_RECORD_PROB} --time-limit {SAMPLE_TIME_LIMIT}

In [ ]:
# Cell 7: Verify Sample Counts
%cd /content/learn2branch-ecole
!echo train samples: $(find data/samples/setcover/500r_1000c_0.05d/train -maxdepth 1 -name 'sample_*.pkl' 2>/dev/null | wc -l)
!echo valid samples: $(find data/samples/setcover/500r_1000c_0.05d/valid -maxdepth 1 -name 'sample_*.pkl' 2>/dev/null | wc -l)
!echo test samples:  $(find data/samples/setcover/500r_1000c_0.05d/test  -maxdepth 1 -name 'sample_*.pkl' 2>/dev/null | wc -l)

In [ ]:
# Cell 8: Train Official Ecole GNN Policy
# This is the GPU-benefiting step. Check model/setcover/0/train_log.txt during/after the run.
%cd /content/learn2branch-ecole
!python 03_train_gnn.py setcover -s {SEED} -g {GPU_ID} --max-epochs {MAX_EPOCHS} --epoch-samples {EPOCH_SAMPLES} --batch-size {BATCH_SIZE} --pretrain-batch-size 128 --valid-batch-size 128

In [ ]:
# Cell 9: Show Training Log and Checkpoint
%cd /content/learn2branch-ecole
!tail -n 80 model/setcover/{SEED}/train_log.txt
!ls -lh model/setcover/{SEED}/train_params.pkl

In [ ]:
# Cell 10: Save Artifacts to Google Drive
# Optional but strongly recommended, because Colab runtimes are ephemeral.
from google.colab import drive
drive.mount('/content/drive')

ARTIFACT_DIR = Path('/content/drive/MyDrive/learn2branch_setcover_100k')
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

!cp /content/learn2branch-ecole/model/setcover/{SEED}/train_params.pkl /content/drive/MyDrive/learn2branch_setcover_100k/train_params_seed{SEED}.pkl
!cp /content/learn2branch-ecole/model/setcover/{SEED}/train_log.txt /content/drive/MyDrive/learn2branch_setcover_100k/train_log_seed{SEED}.txt
!cp -r /content/learn2branch-ecole/data/samples/setcover/500r_1000c_0.05d /content/drive/MyDrive/learn2branch_setcover_100k/samples_500r_1000c_0.05d
!ls -lh /content/drive/MyDrive/learn2branch_setcover_100k